In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

In [2]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    zoom_range=0.3,
    shear_range=0.3,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3]
)

val_datagen = ImageDataGenerator(rescale=1./255)

In [3]:
train_data = train_datagen.flow_from_directory(
    "split_data/train",
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

val_data = val_datagen.flow_from_directory(
    "split_data/val",
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

Found 13726 images belonging to 5 classes.
Found 2722 images belonging to 5 classes.


In [4]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

In [10]:
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 224, 224, 3)]        0         []                            
                                                                                                  
 Conv1 (Conv2D)              (None, 112, 112, 32)         864       ['input_1[0][0]']             
                                                                                                  
 bn_Conv1 (BatchNormalizati  (None, 112, 112, 32)         128       ['Conv1[0][0]']               
 on)                                                                                              
                                                                                                  
 Conv1_relu (ReLU)           (None, 112, 112, 32)         0         ['bn_Conv1[0][0]']        

In [5]:
for layer in base_model.layers:
    layer.trainable = False

In [6]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(train_data.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

In [7]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [8]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(patience=3, restore_best_weights=True)

In [11]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20 
)

Epoch 1/20
429/429 [==============================] - 803s 2s/step - loss: 0.3948 - accuracy: 0.8610 - val_loss: 0.1830 - val_accuracy: 0.9379
Epoch 2/20
429/429 [==============================] - 658s 2s/step - loss: 0.2547 - accuracy: 0.9096 - val_loss: 0.1839 - val_accuracy: 0.9320
Epoch 3/20
429/429 [==============================] - 542s 1s/step - loss: 0.2146 - accuracy: 0.9254 - val_loss: 0.1324 - val_accuracy: 0.9552
Epoch 4/20
429/429 [==============================] - 522s 1s/step - loss: 0.1959 - accuracy: 0.9288 - val_loss: 0.1336 - val_accuracy: 0.9548
Epoch 5/20
429/429 [==============================] - 532s 1s/step - loss: 0.1860 - accuracy: 0.9320 - val_loss: 0.1110 - val_accuracy: 0.9574
Epoch 6/20
429/429 [==============================] - 471s 1s/step - loss: 0.1781 - accuracy: 0.9341 - val_loss: 0.1214 - val_accuracy: 0.9533
Epoch 7/20
429/429 [==============================] - 516s 1s/step - loss: 0.1694 - accuracy: 0.9392 - val_loss: 0.1087 - val_accuracy: 0.9578

In [13]:
model.save("mobilenetv2.h5")

C:\Users\Asus\AppData\Roaming\Python\Python310\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [24]:
import numpy as np
from sklearn.metrics import classification_report

In [21]:
print(classification_report(y_true, y_pred_classes, target_names=class_labels))

              precision    recall  f1-score   support

       Boron       0.10      0.10      0.10       366
     Healthy       0.19      0.19      0.19       463
      Kalium       0.30      0.30      0.30       777
          Mg       0.28      0.28      0.28       734
    nitrogen       0.12      0.12      0.12       382

    accuracy                           0.23      2722
   macro avg       0.20      0.20      0.20      2722
weighted avg       0.22      0.23      0.22      2722



In [22]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred_classes)

print("Confusion Matrix:\n")
print(cm)

Confusion Matrix:

[[ 37  64  99 108  58]
 [ 72  87 124 123  57]
 [100 131 234 204 108]
 [ 99 112 211 209 103]
 [ 61  63 113  99  46]]
